In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Gifts/Travel or Lodging Expenses_Non POs
   
   BUSINESS QUESTION: 
   During the assessment period, how many instances were recorded by the Unit where 
   employees provide or receive gifts... or paid for travel/entertainment to Non-POs
   (Non-Public Officials) greater than $250?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. DATA SOURCE: Uses COUPA FILES ONLY. (Finance files are excluded).
   3. NON-PUBLIC OFFICIAL FILTER: Filters for rows where `PublicOfficial` is 'N' or 'NO'.
   4. THRESHOLD FILTER: Strictly filters for transactions where `Total` > 250.
   5. ACCOUNT PARSING: Parses the 'Account' string to extract the Cost Center and 
      Category Code.
   6. TARGET CATEGORY CODES: Hardcoded explicitly to (066, 009, 012, 067, 095, 134, 
      140, 180, 181, 793).
   7. MAPPING & THE 793 RULE: Maps to AU_IDs. Applies the exception rule that 
      Category Code 793 is ONLY valid for AU 101016.
   8. FINAL OUTPUT: Rolls up and counts the instances, outputting a numerical value.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    -- 2. Union all 7 Coupa files. Includes the Total column for the >250 rule.
    SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Non_PO_Transactions AS (
    -- 3 & 4. Parse Account, filter for NON-Public Officials, and filter Total > 250
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        TRIM(PublicOfficial) AS PublicOfficial,
        -- Safely cast Total to a decimal, removing any potential commas
        TRY_CAST(REPLACE(Total, ',', '') AS DECIMAL(15,2)) AS Transaction_Total
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      -- EBA04 RULE 1: Non-Public Officials Only ('N' or 'NO')
      AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
      -- EBA04 RULE 2: Value must be strictly greater than 250
      AND TRY_CAST(REPLACE(Total, ',', '') AS DECIMAL(15,2)) > 250
),

Flagged_Non_PO_Transactions AS (
    -- 5. Filter against the Hardcoded Category list
    SELECT 
        Cleaned_CC,
        CatCode_Int,
        Transaction_Total
    FROM Coupa_Non_PO_Transactions
    WHERE Cleaned_CC IS NOT NULL
      AND CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793)
),

CC_Mapping AS (
    -- 6a. Standardize the CC Mapping table
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- 6b. Map transactions to AU_IDs, APPLY THE 793 RULE, and Count
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Non_PO_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- THE 793 EXCEPTION RULE: Exclude code 793 if the AU is NOT 101016
        NOT (f.CatCode_Int = 793 AND m.AU_ID != '101016')
    GROUP BY m.AU_ID
)

-- 7. Final Output: Strictly structured numerical count anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA04' AS QuestionID,               
    -- Numerical Output: Counts the instances. If none, outputs '0'
    COALESCE(CAST(f.Flagged_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA04 - Gifts/Travel/Lodging (Non-POs > $250) Traceability
   
   PURPOSE: 
   Isolates the data flow for EBA04 to show exactly how Coupa records are evaluated
   against the > 250 threshold, the Non-PO ('N'/'NO') flag, the Target Categories, 
   and the strict "793" mapping rule.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Combined_Coupa_Raw AS (
    -- Union all 7 Coupa files
    SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- Parse the Coupa string into CC, Category Code, and Total
    SELECT 
        Account AS Raw_String,
        UPPER(TRIM(PublicOfficial)) AS Raw_PO_Flag,
        TRY_CAST(REPLACE(Total, ',', '') AS DECIMAL(15,2)) AS Parsed_Total,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    -- Grab Bootstrap mappings
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    'Coupa' AS Source_System,
    c.Raw_String AS Original_Record,
    c.Raw_PO_Flag AS Non_Public_Official_Flag,
    c.Parsed_Total,
    
    -- EBA04 Condition 1: Non-PO
    CASE WHEN c.Raw_PO_Flag IN ('N', 'NO') THEN '✅ PASS' ELSE '❌ FAIL' END AS Non_PO_Condition,
    
    -- EBA04 Condition 2: Total > 250
    CASE WHEN c.Parsed_Total > 250 THEN '✅ PASS' ELSE '❌ FAIL (<= 250 or Null)' END AS Threshold_Condition,
    
    c.Cleaned_CC AS Parsed_Cost_Center,
    c.CatCode_Int AS Parsed_Category_Code,
    
    -- EBA04 Condition 3: Hit the Master target list
    CASE WHEN c.CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793) THEN '✅ PASS' ELSE '❌ FAIL' END AS Category_Condition,
         
    m.AU_ID AS Mapped_AU_ID,
    mast.Master_AU_Name,
    
    -- Trace exactly how the transaction navigates the rules
    CASE 
        WHEN c.Raw_PO_Flag NOT IN ('N', 'NO') THEN '❌ DROPPED: Is a Public Official'
        WHEN c.Parsed_Total <= 250 OR c.Parsed_Total IS NULL THEN '❌ DROPPED: Total not > 250'
        WHEN c.CatCode_Int NOT IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793) THEN '❌ DROPPED: Not a target category'
        WHEN c.CatCode_Int = 793 AND m.AU_ID != '101016' THEN '❌ BLOCKED: 793 only valid for 101016'
        WHEN c.CatCode_Int = 793 AND m.AU_ID = '101016' THEN '✅ KEPT: Valid 793 mapping > $250'
        ELSE '✅ KEPT: Standard Category > $250'
    END AS Pipeline_Status
    
FROM Coupa_Parsed c
LEFT JOIN CC_Mapping m 
    ON c.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.Master_Numeric_ID

-- Optional: Uncomment to view only the rows that passed the > 250 threshold
-- WHERE c.Parsed_Total > 250
ORDER BY c.Cleaned_CC;